# exp01b - training speed with AMP (fp16) + no UNet gradient checkpointing

Same 150-iteration profile as exp01 (3.4 s/it baseline). Measures speed-up and peak memory on 2x T4.


In [ ]:
import os
os.environ["BIOHUB_PROF_EXTRA_FLAGS"] = "--amp --no-grad-ckpt"
os.environ["BIOHUB_PROF_MAX_ITERS"] = "150"
print("profile flags:", os.environ["BIOHUB_PROF_EXTRA_FLAGS"])


In [ ]:
# --- Setup header: constants expected by the dependency/repo helpers (from baseline cell 8) ---
from __future__ import annotations
import csv, importlib.util, json, math, os, shutil, subprocess, tempfile, zipfile, sys, time
from pathlib import Path

COMPETITION = "biohub-cell-tracking-during-development"
COMP_DIR_CANDIDATES = [Path(f"/kaggle/input/competitions/{COMPETITION}"), Path(f"/kaggle/input/{COMPETITION}")]
COMP_DIR = next((p for p in COMP_DIR_CANDIDATES if p.exists()), COMP_DIR_CANDIDATES[0])
TRAIN_DIR = COMP_DIR / "train"
TEST_DIR = COMP_DIR / "test"
WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
REPO_DIR = WORKING_DIR / "tracking_repo"
METHOD = "unet_transformer"
WEIGHTS_RELATIVE = f"weights/{METHOD}/split_0/edge_predictor_best.pth"
TARGET_ARTIFACT_SLUG = os.environ.get("BIOHUB_TARGET_ARTIFACT_SLUG", "biohub-tracking-support-pack-50ep-v1")
PRIMARY_ARTIFACT_MANIFEST = Path(os.environ.get(
    "BIOHUB_PRIMARY_ARTIFACT_MANIFEST",
    "/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/ARTIFACT_MANIFEST.json"))
ALLOW_ARTIFACT_FALLBACK = os.environ.get("BIOHUB_ALLOW_ARTIFACT_FALLBACK", "0") != "0"


In [ ]:
import re

os.environ.setdefault("POLARS_PREFER_PKG", "32")

PACKAGE_SPECS = {
    "tracksdata": ("tracksdata", "tracksdata"),
    "zarr": ("zarr", "zarr>=3.0.10,<4"),
    "pyscipopt": ("pyscipopt", "pyscipopt"),
    "geff": ("geff", "geff>=1.1.3.1.1"),
    "geff_spec": ("geff_spec", "geff-spec<1.2"),
    "ilpy": ("ilpy", "ilpy>=0.5.1"),
    "polars": ("polars", "polars>=1.36"),
    "blosc2": ("blosc2", "blosc2"),
    "dask": ("dask", "dask"),
    "imagecodecs": ("imagecodecs", "imagecodecs"),
    "skimage": ("skimage", "scikit-image>=0.24"),
    "pyarrow": ("pyarrow", "pyarrow"),
    "rustworkx": ("rustworkx", "rustworkx>=0.17.1"),
    "sqlalchemy": ("sqlalchemy", "sqlalchemy>=2"),
    "numcodecs": ("numcodecs", "numcodecs>=0.13,<0.16"),
    "donfig": ("donfig", "donfig>=0.8"),
    "google_crc32c": ("google_crc32c", "google-crc32c>=1.5"),
    "bidict": ("bidict", "bidict>=0.23.1"),
    "psygnal": ("psygnal", "psygnal>=0.14"),
    "rich": ("rich", "rich"),
    "networkx": ("networkx", "networkx>=3.2.1"),
    "pydantic": ("pydantic", "pydantic>=2.11"),
    "pydantic_core": ("pydantic_core", "pydantic-core"),
    "annotated_types": ("annotated_types", "annotated-types"),
    "typing_extensions": ("typing_extensions", "typing-extensions>=4.13"),
    "typing_inspection": ("typing_inspection", "typing-inspection"),
    "markdown_it": ("markdown_it", "markdown-it-py"),
    "pygments": ("pygments", "pygments"),
    "click": ("click", "click"),
    "cloudpickle": ("cloudpickle", "cloudpickle"),
    "fsspec": ("fsspec", "fsspec"),
    "partd": ("partd", "partd"),
    "locket": ("locket", "locket"),
    "toolz": ("toolz", "toolz"),
    "yaml": ("yaml", "pyyaml"),
    "ndindex": ("ndindex", "ndindex"),
    "msgpack": ("msgpack", "msgpack"),
    "numexpr": ("numexpr", "numexpr"),
    "deprecated": ("deprecated", "deprecated"),
    "wrapt": ("wrapt", "wrapt"),
    "imageio": ("imageio", "imageio"),
    "PIL": ("PIL", "pillow"),
    "tifffile": ("tifffile", "tifffile"),
    "lazy_loader": ("lazy_loader", "lazy-loader"),
    "tqdm": ("tqdm", "tqdm"),
}
EXTRA_SPECS_BY_NAME = {
    "tracksdata": ["bidict>=0.23.1", "psygnal>=0.14", "rich"],
    "zarr": ["donfig>=0.8", "google-crc32c>=1.5", "numcodecs>=0.13,<0.16"],
    "geff": ["geff-spec<1.2", "networkx>=3.2.1", "pydantic>=2.11", "numcodecs>=0.13,<0.16"],
    "geff_spec": ["pydantic>=2.11", "annotated-types", "pydantic-core", "typing-inspection"],
    "polars": ["polars-runtime-32"],
    "dask": ["click", "cloudpickle", "fsspec", "partd", "pyyaml", "toolz"],
    "partd": ["locket"],
    "blosc2": ["ndindex", "msgpack", "numexpr"],
    "numcodecs": ["deprecated", "msgpack", "wrapt"],
    "rich": ["markdown-it-py", "pygments"],
    "pydantic": ["annotated-types", "pydantic-core", "typing-extensions>=4.13", "typing-inspection"],
    "skimage": ["imageio", "pillow", "tifffile", "lazy-loader", "networkx"],
}
PIP_DEPENDENCIES = [spec for _, spec in PACKAGE_SPECS.values()]
REQUIRED_MODULES = {name: module for name, (module, _) in PACKAGE_SPECS.items() if module}
FALLBACK_ARTIFACT_SLUGS = ["biohub-tracking-support-pack-v1"]
ALLOW_PIP_INSTALL = os.environ.get("BIOHUB_ALLOW_PIP_INSTALL", "0") != "0"


def module_missing(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is None


def has_model_artifact(path: Path) -> bool:
    has_repo_dir = (path / "repo").exists()
    has_weights_dir = (path / "weights" / METHOD / "split_0" / "edge_predictor_best.pth").exists()
    has_repo_zip = (path / "repo.zip").exists()
    has_weights_zip = (path / "weights.zip").exists()
    return (has_repo_dir and has_weights_dir) or (has_repo_zip and has_weights_zip)


def artifact_manifest(path: Path) -> dict:
    manifest = path / "ARTIFACT_MANIFEST.json"
    if not manifest.exists():
        return {}
    try:
        return json.loads(manifest.read_text())
    except Exception:
        return {}


def artifact_matches_target(path: Path) -> bool:
    if ALLOW_ARTIFACT_FALLBACK:
        return True
    manifest = artifact_manifest(path)
    artifact_name = str(manifest.get("artifact_name", ""))
    path_text = str(path)
    return TARGET_ARTIFACT_SLUG in {artifact_name, path.name} or TARGET_ARTIFACT_SLUG in path_text


def candidate_roots_for_slug(slug: str) -> list[Path]:
    return [
        Path(f"/kaggle/input/datasets/pilkwang/{slug}"),
        Path(f"/kaggle/input/{slug}"),
        Path(f"/kaggle/input/{slug}/{slug}"),
        Path(f"PublicNotebook/{slug}"),
    ]


def find_artifacts_root() -> Path:
    candidates: list[Path] = []
    for env_name in ["BIOHUB_MODEL_ARTIFACTS", "BIOHUB_ARTIFACTS"]:
        explicit = os.environ.get(env_name, "").strip()
        if explicit:
            candidates.append(Path(explicit))

    candidates.append(PRIMARY_ARTIFACT_MANIFEST.parent)
    candidates.extend(candidate_roots_for_slug(TARGET_ARTIFACT_SLUG))

    if ALLOW_ARTIFACT_FALLBACK:
        for slug in FALLBACK_ARTIFACT_SLUGS:
            candidates.extend(candidate_roots_for_slug(slug))

    input_root = Path("/kaggle/input")
    if input_root.exists():
        for child in input_root.iterdir():
            if not child.is_dir():
                continue
            child_text = str(child)
            if TARGET_ARTIFACT_SLUG in child_text or ALLOW_ARTIFACT_FALLBACK:
                candidates.append(child)
                candidates.append(child / child.name)
                for grandchild in child.iterdir():
                    if grandchild.is_dir():
                        candidates.append(grandchild)

    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.expanduser()
        if candidate in seen:
            continue
        seen.add(candidate)
        if has_model_artifact(candidate) and artifact_matches_target(candidate):
            return candidate
    checked = "\n".join(str(path) for path in candidates[:80])
    raise FileNotFoundError(
        "Could not find the required model artifact. "
        f"Expected slug: {TARGET_ARTIFACT_SLUG}\n"
        "Attach the newly uploaded support dataset, or set BIOHUB_MODEL_ARTIFACTS.\n"
        "To debug with an older artifact, set BIOHUB_ALLOW_ARTIFACT_FALLBACK=1.\n"
        "Checked:\n" + checked
    )


def _has_package_file(path: Path) -> bool:
    if not path.exists() or not path.is_dir():
        return False
    patterns = ("*.whl", "*.tar.gz", "*.zip")
    return any(any(path.glob(pattern)) for pattern in patterns)


def find_offline_package_dirs(artifacts: Path) -> list[Path]:
    candidates: list[Path] = [
        artifacts / "wheels",
        artifacts,
        Path("/kaggle/working"),
        Path("/kaggle/working/wheels"),
    ]
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for child in input_root.iterdir():
            if child.is_dir():
                candidates.extend([child / "wheels", child])
                for grandchild in child.iterdir():
                    if grandchild.is_dir():
                        candidates.extend([grandchild / "wheels", grandchild])

    out: list[Path] = []
    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.expanduser()
        if candidate in seen:
            continue
        seen.add(candidate)
        if _has_package_file(candidate):
            out.append(candidate)
    return out


def purge_imported_modules(package_names: list[str]) -> None:
    roots = {"tracksdata"}
    for name in package_names:
        if name in PACKAGE_SPECS:
            module = PACKAGE_SPECS[name][0]
            roots.add(module.split(".")[0])
        if name == "polars":
            roots.add("polars")
    for root in roots:
        for module_name in list(sys.modules):
            if module_name == root or module_name.startswith(root + "."):
                sys.modules.pop(module_name, None)


def polars_runtime_ready() -> bool:
    try:
        import polars as _pl
        from polars._plr import PySeries as _PySeries

        _ = _PySeries
        return hasattr(_pl, "Float16") and _pl.Series([-999999.0], dtype=_pl.Float64).dtype == _pl.Float64
    except Exception:
        return False


def packages_requiring_refresh() -> list[str]:
    refresh: list[str] = []
    if not module_missing("polars") and not polars_runtime_ready():
        refresh.append("polars")

    if not module_missing("zarr"):
        try:
            import zarr as _zarr
            version_text = str(getattr(_zarr, "__version__", "0"))
            major = int(version_text.split(".", 1)[0])
            if major < 3:
                refresh.append("zarr")
        except Exception:
            refresh.append("zarr")
    return refresh


def dependency_specs_for(missing: list[str]) -> list[str]:
    specs: list[str] = []
    seen: set[str] = set()

    def add(spec: str) -> None:
        key = spec.lower()
        if key not in seen:
            seen.add(key)
            specs.append(spec)

    for name in missing:
        if name in PACKAGE_SPECS:
            add(PACKAGE_SPECS[name][1])
        for spec in EXTRA_SPECS_BY_NAME.get(name, []):
            add(spec)
    return specs


def import_failures() -> dict[str, str]:
    failures: dict[str, str] = {}
    for name, module_name in REQUIRED_MODULES.items():
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            failures[name] = f"{type(exc).__name__}: {exc}"
    return failures


def missing_names_from_failures(failures: dict[str, str]) -> list[str]:
    names: list[str] = []
    module_to_name = {module: name for name, module in REQUIRED_MODULES.items()}
    for message in failures.values():
        match = re.search(r"No module named ['\"]([^'\"]+)['\"]", message)
        if match:
            module = match.group(1).split(".")[0]
        else:
            match = re.search(r"module ['\"]([^'\"]+)['\"] has no attribute", message)
            if not match:
                continue
            module = match.group(1).split(".")[0]
        name = module_to_name.get(module)
        if name and name not in names:
            names.append(name)
    return names


def install_missing_dependencies(missing: list[str], artifacts: Path) -> None:
    specs = dependency_specs_for(missing)
    force_reinstall = bool({"polars", "zarr"} & set(missing))
    if not specs:
        return

    package_dirs = find_offline_package_dirs(artifacts)
    if package_dirs:
        offline_cmd = [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps"]
        if force_reinstall:
            offline_cmd.append("--force-reinstall")
        for package_dir in package_dirs:
            offline_cmd.extend(["--find-links", str(package_dir)])
        offline_cmd.extend(specs)
        print("Installing missing packages from offline package dirs:", missing)
        print("Dependency resolver is disabled with --no-deps to avoid replacing Kaggle numpy/scipy in a live kernel.")
        print("Offline package dirs:", [str(path) for path in package_dirs])
        result = subprocess.run(offline_cmd, text=True, capture_output=True)
        if result.returncode == 0:
            purge_imported_modules(missing)
            print("Offline dependency install succeeded.")
            return
        print("Offline dependency install failed. Last pip output:")
        print((result.stdout or "")[-2000:])
        print((result.stderr or "")[-2000:])

    if ALLOW_PIP_INSTALL:
        online_cmd = [sys.executable, "-m", "pip", "install", "--no-deps"]
        if force_reinstall:
            online_cmd.append("--force-reinstall")
        online_cmd.extend(specs)
        print("Installing missing packages from PyPI:", missing)
        result = subprocess.run(online_cmd, text=True, capture_output=True)
        if result.returncode == 0:
            purge_imported_modules(missing)
            print("PyPI dependency install succeeded.")
            return
        print("PyPI dependency install failed. Last pip output:")
        print((result.stdout or "")[-2000:])
        print((result.stderr or "")[-2000:])

    command = "pip install tracksdata zarr>=3.0.10,<4 pyscipopt geff geff-spec ilpy polars blosc2 dask imagecodecs pyarrow rustworkx sqlalchemy donfig numcodecs"
    raise ImportError(
        "Missing required packages or dependency wheels: " + ", ".join(missing) + "\n"
        "Attach the support dataset with offline wheels. If supplying Kaggle dependency input instead, use:\n"
        + command + "\n"
        "Do not quote zarr>=3.0.10,<4 in Kaggle dependency input."
    )


def ensure_dependencies(artifacts: Path) -> None:
    for _ in range(5):
        refresh = packages_requiring_refresh()
        if refresh:
            install_missing_dependencies(refresh, artifacts)
            continue

        missing = [pkg for pkg, module in REQUIRED_MODULES.items() if module_missing(module)]
        if missing:
            install_missing_dependencies(missing, artifacts)
            continue

        failures = import_failures()
        if not failures:
            print("Required graph/Zarr/ILP packages import successfully.")
            return

        missing_from_import = missing_names_from_failures(failures)
        if missing_from_import:
            install_missing_dependencies(missing_from_import, artifacts)
            continue

        raise ImportError(
            "Required packages are present but failed to import. "
            "This may indicate a binary dependency mismatch in the live notebook kernel. "
            "Keep Kaggle dependency input empty and attach the wheels artifact.\n"
            + json.dumps(failures, indent=2)
        )

    failures = import_failures()
    raise ImportError(
        "Dependency recovery did not converge after repeated offline installs. "
        "The attached support artifact may be missing wheels.\n"
        + json.dumps(failures, indent=2)
    )


def remove_path(path: Path) -> None:
    if path.is_symlink() or path.is_file():
        path.unlink()
    elif path.exists():
        shutil.rmtree(path)


def copy_or_extract_tree(src_dir: Path, src_zip: Path, dst: Path) -> None:
    remove_path(dst)
    if src_dir.exists() and src_dir.is_dir():
        shutil.copytree(src_dir, dst)
        return
    if src_zip.exists() and src_zip.is_file():
        dst.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(src_zip) as zf:
            zf.extractall(dst)
        return
    raise FileNotFoundError(f"Missing source tree or zip: {src_dir} / {src_zip}")


def link_or_copy_tree(src: Path, dst: Path) -> None:
    remove_path(dst)
    try:
        os.symlink(src, dst, target_is_directory=True)
    except Exception:
        shutil.copytree(src, dst)


def materialize_inference_repo(artifacts: Path) -> None:
    copy_or_extract_tree(artifacts / "repo", artifacts / "repo.zip", REPO_DIR)

    weights_src = artifacts / "weights"
    weights_zip = artifacts / "weights.zip"
    weights_dst = REPO_DIR / "weights"
    if weights_src.exists() and weights_src.is_dir():
        link_or_copy_tree(weights_src, weights_dst)
    elif weights_zip.exists() and weights_zip.is_file():
        remove_path(weights_dst)
        weights_dst.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(weights_zip) as zf:
            zf.extractall(weights_dst)
    else:
        raise FileNotFoundError(f"Missing weights tree or zip under {artifacts}")

    required = [
        REPO_DIR / "scripts" / "predict_unet_transformer.py",
        REPO_DIR / WEIGHTS_RELATIVE,
    ]
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError("Materialized inference repo is incomplete:\n" + "\n".join(missing))
    print("Inference repo:", REPO_DIR)
    print("Weights:", REPO_DIR / WEIGHTS_RELATIVE)


ARTIFACTS = find_artifacts_root()
print("ARTIFACTS:", ARTIFACTS)
print("Has offline wheels:", (ARTIFACTS / "wheels").exists())
manifest_info = artifact_manifest(ARTIFACTS)
if manifest_info:
    print("Artifact name:", manifest_info.get("artifact_name"))
    print("Weight sha256:", manifest_info.get("model", {}).get("weight_sha256"))
    print("Weight path:", manifest_info.get("model", {}).get("weight_path"))
    _expected_primary_sha256 = "12f6881ee3620a831697ca098ff8f48e687a24225f4e048b538deec3562fe771"
    _actual_primary_sha256 = str(manifest_info.get("model", {}).get("weight_sha256", ""))
    if _actual_primary_sha256 != _expected_primary_sha256:
        raise RuntimeError(
            "Primary model checksum mismatch: "
            f"expected {_expected_primary_sha256}, got {_actual_primary_sha256 or 'missing'}"
        )

ensure_dependencies(ARTIFACTS)
materialize_inference_repo(ARTIFACTS)


In [ ]:
# --- Patches for scripts/train_unet_transformer.py (applied at runtime inside the Kaggle notebook) ---
# Adds:
#   --init-weights PATH        : initialise the full UNet+transformer from a checkpoint (fine-tuning)
#   --det-loss {bce,gauss_focal}: detection loss variant. gauss_focal = anisotropic Gaussian heatmap
#                                targets + CenterNet-style penalty-reduced focal loss (PAC-MAP / nnLandmark ideas)
#   --det-sigma-um FLOAT       : Gaussian sigma in microns for gauss_focal (default 2.5)
#   --seed INT                 : torch/numpy seed
# Usage in notebook:  apply_train_patches(TRAIN_REPO / "scripts" / "train_unet_transformer.py")
from pathlib import Path


def _replace_once(src: str, old: str, new: str, label: str) -> str:
    n = src.count(old)
    if n != 1:
        raise RuntimeError(f"patch '{label}': expected exactly 1 occurrence, found {n}")
    return src.replace(old, new)


GAUSS_LOSS_CODE = '''
# ---- exp patches: detection loss variants -----------------------------------
DET_LOSS_MODE = "bce"
DET_SIGMA_UM = 2.5


def _gaussian_1d(sigma_vox: float, device) -> torch.Tensor:
    sigma_vox = max(float(sigma_vox), 1e-3)
    r = max(1, int(round(3.0 * sigma_vox)))
    x = torch.arange(-r, r + 1, device=device, dtype=torch.float32)
    k = torch.exp(-0.5 * (x / sigma_vox) ** 2)
    return k / k.max()  # peak = 1 so a lone point keeps target 1 at its centre


def _gaussian_heatmap(points: torch.Tensor, voxel_size, sigma_um: float) -> torch.Tensor:
    """points: (B, Z, Y, X) binary. Returns soft target in [0, 1] with anisotropic Gaussian blobs."""
    B = points.shape[0]
    h = points[:, None].float()  # (B,1,Z,Y,X)
    for axis, vs in enumerate(voxel_size):
        k = _gaussian_1d(sigma_um / float(vs), points.device)
        r = k.numel() // 2
        shape = [1, 1, 1, 1, 1]
        shape[2 + axis] = k.numel()
        pad = [0, 0, 0, 0, 0, 0]
        pad[2 * (2 - axis)] = r
        pad[2 * (2 - axis) + 1] = r
        h = F.conv3d(F.pad(h, pad), k.reshape(shape))
    return h[:, 0].clamp(max=1.0)


def compute_detection_loss_gauss_focal(
    det_logits: torch.Tensor,
    coords: torch.Tensor,
    mask: torch.Tensor,
    neg_weight: float,
    voxel_size,
    sigma_um: float,
    alpha: float = 2.0,
    beta: float = 4.0,
) -> torch.Tensor:
    """CenterNet-style loss on Gaussian heatmap targets, per-sample normalised like the BCE variant.

    pos (target == 1):   (1-p)^alpha * -log(p)                  / n_pos
    neg (target  < 1):   (1-t)^beta * p^alpha * -log(1-p)       * neg_weight / n_neg
    The neg_weight keeps the positive-unlabeled down-weighting of unannotated nuclei.
    """
    B = det_logits.shape[0]
    spatial = det_logits.shape[2:]
    logits = det_logits[:, 0]
    points = torch.zeros_like(logits)
    nt = mask.sum(dim=1).long()
    for b in range(B):
        n_gt = int(nt[b])
        if n_gt <= 0:
            continue
        gt = coords[b, :n_gt]
        zi = gt[:, 0].long().clamp(0, spatial[0] - 1)
        yi = gt[:, 1].long().clamp(0, spatial[1] - 1)
        xi = gt[:, 2].long().clamp(0, spatial[2] - 1)
        points[b, zi, yi, xi] = 1.0
    with torch.no_grad():
        target = _gaussian_heatmap(points, voxel_size, sigma_um)
    p = torch.sigmoid(logits.float()).clamp(1e-6, 1 - 1e-6)
    pos = points > 0.5
    pos_term = ((1 - p) ** alpha) * (-torch.log(p))
    neg_term = ((1 - target) ** beta) * (p ** alpha) * (-torch.log(1 - p))
    n_pos = pos.reshape(B, -1).sum(dim=1).clamp(min=1).float()
    n_neg = (pos.numel() // B - n_pos).clamp(min=1)
    pos_loss = (pos_term * pos).reshape(B, -1).sum(dim=1) / n_pos
    neg_loss = (neg_term * (~pos)).reshape(B, -1).sum(dim=1) / n_neg * neg_weight
    return (pos_loss + neg_loss).mean()


'''

PATCHES = [
    # --- gauss focal loss: definitions inserted before the original compute_detection_loss
    (
        "def compute_detection_loss(\n    det_logits: torch.Tensor,\n    coords: torch.Tensor,\n    mask: torch.Tensor,\n    neg_weight: float = 0.1,\n) -> torch.Tensor:\n",
        GAUSS_LOSS_CODE
        + "def compute_detection_loss(\n    det_logits: torch.Tensor,\n    coords: torch.Tensor,\n    mask: torch.Tensor,\n    neg_weight: float = 0.1,\n    voxel_size=None,\n) -> torch.Tensor:\n"
        + "    if DET_LOSS_MODE == \"gauss_focal\" and voxel_size is not None:\n"
        + "        return compute_detection_loss_gauss_focal(det_logits, coords, mask, neg_weight, voxel_size, DET_SIGMA_UM)\n",
        "det-loss dispatch",
    ),
    # --- pass voxel_size from train_epoch
    (
        "            compute_detection_loss(\n                det_logits[i], coords[:, i], masks[:, i],\n                det_neg_weight,\n            )\n            for i in range(W)\n",
        "            compute_detection_loss(\n                det_logits[i], coords[:, i], masks[:, i],\n                det_neg_weight, voxel_size=voxel_size,\n            )\n            for i in range(W)\n",
        "det-loss voxel_size",
    ),
    # --- init weights: train() signature
    (
        "    pool_kernel_um: float = 5.0,\n    data_parallel: bool = True,\n) -> UNetNodeTransformer:\n",
        "    pool_kernel_um: float = 5.0,\n    data_parallel: bool = True,\n    init_weights: Path | None = None,\n) -> UNetNodeTransformer:\n",
        "init-weights signature",
    ),
    # --- init weights: load after model creation (before DataParallel wrap)
    (
        "    model = UNetNodeTransformer(\n        unet=unet,\n        unet_out_channels=unet_out_channels,\n        pos_feat_dim=pos_feat_dim,\n    ).to(device)\n",
        "    model = UNetNodeTransformer(\n        unet=unet,\n        unet_out_channels=unet_out_channels,\n        pos_feat_dim=pos_feat_dim,\n    ).to(device)\n"
        "    if init_weights is not None:\n"
        "        _state = torch.load(init_weights, map_location=\"cpu\", weights_only=True)\n"
        "        _state = {k.replace(\"unet.module.\", \"unet.\", 1): v for k, v in _state.items()}\n"
        "        _missing, _unexpected = model.load_state_dict(_state, strict=False)\n"
        "        print(f\"  init weights {init_weights}: {len(_missing)} missing, {len(_unexpected)} unexpected\", flush=True)\n",
        "init-weights load",
    ),
    # --- CLI args
    (
        "    parser.add_argument(\"--single-gpu\", dest=\"data_parallel\", action=\"store_false\",\n                        help=\"Disable multi-GPU; train on cuda:0 only.\")\n",
        "    parser.add_argument(\"--single-gpu\", dest=\"data_parallel\", action=\"store_false\",\n                        help=\"Disable multi-GPU; train on cuda:0 only.\")\n"
        "    parser.add_argument(\"--init-weights\", type=str, default=None, help=\"Full model checkpoint to fine-tune from.\")\n"
        "    parser.add_argument(\"--det-loss\", type=str, default=\"bce\", choices=[\"bce\", \"gauss_focal\"])\n"
        "    parser.add_argument(\"--det-sigma-um\", type=float, default=2.5)\n"
        "    parser.add_argument(\"--seed\", type=int, default=None)\n",
        "cli args",
    ),
    # --- apply CLI args in main
    (
        "    args = parser.parse_args()\n\n    from dataspec import DATASET_PATH\n",
        "    args = parser.parse_args()\n"
        "    global DET_LOSS_MODE, DET_SIGMA_UM\n"
        "    DET_LOSS_MODE = args.det_loss\n"
        "    DET_SIGMA_UM = args.det_sigma_um\n"
        "    print(f\"det loss mode={DET_LOSS_MODE} sigma_um={DET_SIGMA_UM} init_weights={args.init_weights} seed={args.seed}\", flush=True)\n"
        "    if args.seed is not None:\n"
        "        torch.manual_seed(args.seed); np.random.seed(args.seed)\n\n"
        "    from dataspec import DATASET_PATH\n",
        "main apply args",
    ),
    # --- pass init_weights + seed into train()
    (
        "        train(\n            data_dir=data_dir,\n            fold=fold,\n",
        "        train(\n            init_weights=Path(args.init_weights) if args.init_weights else None,\n            seed=args.seed,\n            data_dir=data_dir,\n            fold=fold,\n",
        "train() call",
    ),
    # ================= speed: AMP (fp16 autocast + GradScaler) and optional no-grad-checkpointing =================
    (
        "DET_LOSS_MODE = \"bce\"\nDET_SIGMA_UM = 2.5\n",
        "DET_LOSS_MODE = \"bce\"\nDET_SIGMA_UM = 2.5\nUSE_AMP = False\nNO_GRAD_CKPT = False\nSCALER = torch.amp.GradScaler(\"cuda\", enabled=False)\n",
        "amp globals",
    ),
    (
        "    parser.add_argument(\"--seed\", type=int, default=None)\n",
        "    parser.add_argument(\"--seed\", type=int, default=None)\n"
        "    parser.add_argument(\"--amp\", action=\"store_true\", help=\"fp16 autocast for UNet/transformer forward + GradScaler\")\n"
        "    parser.add_argument(\"--no-grad-ckpt\", action=\"store_true\", help=\"disable UNet gradient checkpointing (faster, more memory)\")\n",
        "amp cli",
    ),
    (
        "    global DET_LOSS_MODE, DET_SIGMA_UM\n",
        "    global DET_LOSS_MODE, DET_SIGMA_UM, USE_AMP, NO_GRAD_CKPT, SCALER\n"
        "    USE_AMP = bool(args.amp); NO_GRAD_CKPT = bool(args.no_grad_ckpt)\n"
        "    SCALER = torch.amp.GradScaler(\"cuda\", enabled=USE_AMP)\n"
        "    print(f\"amp={USE_AMP} no_grad_ckpt={NO_GRAD_CKPT}\", flush=True)\n",
        "amp apply",
    ),
    # wrap encode/predict_edges in autocast; disable checkpointing; placed right after init-weights loading
    (
        "        print(f\"  init weights {init_weights}: {len(_missing)} missing, {len(_unexpected)} unexpected\", flush=True)\n",
        "        print(f\"  init weights {init_weights}: {len(_missing)} missing, {len(_unexpected)} unexpected\", flush=True)\n"
        "    if NO_GRAD_CKPT:\n"
        "        model.unet.gradient_checkpointing = False\n"
        "        print(\"  UNet gradient checkpointing disabled\", flush=True)\n"
        "    if USE_AMP:\n"
        "        _enc_orig, _pe_orig = model.encode, model.predict_edges\n"
        "        def _enc_amp(*a, **k):\n"
        "            with torch.autocast(\"cuda\", dtype=torch.float16):\n"
        "                return _enc_orig(*a, **k)\n"
        "        def _pe_amp(*a, **k):\n"
        "            with torch.autocast(\"cuda\", dtype=torch.float16):\n"
        "                return _pe_orig(*a, **k)\n"
        "        model.encode, model.predict_edges = _enc_amp, _pe_amp\n"
        "        print(\"  AMP autocast enabled for encode/predict_edges\", flush=True)\n",
        "amp wrap",
    ),
    # losses in fp32
    (
        "    probs = torch.softmax(logits, dim=0)  # dim=0 intentional: divisions allowed, merges aren't\n",
        "    probs = torch.softmax(logits.float(), dim=0)  # dim=0 intentional: divisions allowed, merges aren't\n",
        "edge loss fp32",
    ),
    (
        "    logits = det_logits[:, 0]  # (B, Z, Y, X)\n    target = torch.zeros_like(logits)\n",
        "    logits = det_logits[:, 0].float()  # (B, Z, Y, X)\n    target = torch.zeros_like(logits)\n",
        "det loss fp32",
    ),
    # scaler-aware backward
    (
        "        optimizer.zero_grad()\n        loss.backward()\n        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n        optimizer.step()\n",
        "        optimizer.zero_grad()\n        SCALER.scale(loss).backward()\n        SCALER.unscale_(optimizer)\n        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n        SCALER.step(optimizer)\n        SCALER.update()\n",
        "scaler backward",
    ),
]


def apply_train_patches(script_path: Path) -> None:
    script_path = Path(script_path)
    src = script_path.read_text(encoding="utf-8")
    for old, new, label in PATCHES:
        src = _replace_once(src, old, new, label)
    script_path.write_text(src, encoding="utf-8")
    print(f"applied {len(PATCHES)} train patches to {script_path}")


In [ ]:
# --- Profiling run of the training recipe ---
import json, os, subprocess, sys, time, shutil
from pathlib import Path

EPOCHS = int(os.environ.get("BIOHUB_PROF_EPOCHS", "1"))
MAX_ITERS = int(os.environ.get("BIOHUB_PROF_MAX_ITERS", "150"))
BATCH = int(os.environ.get("BIOHUB_PROF_BATCH", "8"))
N_TRAIN = int(os.environ.get("BIOHUB_PROF_N_TRAIN", "20"))
N_VAL = int(os.environ.get("BIOHUB_PROF_N_VAL", "2"))
WINDOW = int(os.environ.get("BIOHUB_PROF_WINDOW", "2"))
DOWNSAMPLE = os.environ.get("BIOHUB_PROF_DOWNSAMPLE", "1,4,4")

# Writable training copy of the repo: REPO_DIR/weights is a symlink into read-only /kaggle/input.
TRAIN_REPO = WORKING_DIR / "train_repo"
shutil.rmtree(TRAIN_REPO, ignore_errors=True)
TRAIN_REPO.mkdir(parents=True)
for sub in ("scripts", "src"):
    shutil.copytree(REPO_DIR / sub, TRAIN_REPO / sub)
(TRAIN_REPO / "weights").mkdir()
shutil.copytree(REPO_DIR / "weights" / METHOD, TRAIN_REPO / "weights" / METHOD)  # pretrained init, if needed
print("train repo:", TRAIN_REPO, sorted(p.name for p in TRAIN_REPO.iterdir()))
apply_train_patches(TRAIN_REPO / "scripts" / "train_unet_transformer.py")

stems = sorted(p.name[:-5] for p in TRAIN_DIR.iterdir() if p.name.endswith(".zarr"))
test_stems = {p.name[:-5] for p in TEST_DIR.iterdir() if p.name.endswith(".zarr")}
stems = [s for s in stems if s not in test_stems]
a = [s for s in stems if s.startswith("44b6")]
b = [s for s in stems if s.startswith("6bba")]
train = a[: N_TRAIN // 2] + b[: N_TRAIN // 2]
val = a[-N_VAL:] + b[-N_VAL:]
splits = [{"split": 0, "train": train, "test": val}]
splits_path = TRAIN_REPO / "prof_splits.json"
splits_path.write_text(json.dumps(splits, indent=1))
print("train", len(train), "val", len(val))

cmd = [sys.executable, "scripts/train_unet_transformer.py",
       "--method", "prof",
       "--data-dir", str(TRAIN_DIR),
       "--splits", str(splits_path),
       "--split", "0",
       "--epochs", str(EPOCHS),
       "--max-iters", str(MAX_ITERS),
       "--batch-size", str(BATCH),
       "--num-workers", "4",
       "--det-loss-weight", "1.0",
       "--det-neg-weight", "0.01",
       "--window-size", str(WINDOW),
       "--downsample", DOWNSAMPLE,
       "--pool-kernel-um", "5.0"]
for flag in os.environ.get("BIOHUB_PROF_EXTRA_FLAGS", "").split():
    cmd.append(flag)
print(" ".join(cmd), flush=True)
env = {**os.environ, "PYTHONPATH": "src", "PYTHONUNBUFFERED": "1"}
t0 = time.time()
log_path = WORKING_DIR / "train_profile.log"
with open(log_path, "w") as log:
    proc = subprocess.Popen(cmd, cwd=TRAIN_REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="", flush=True)
        log.write(line)
    rc = proc.wait()
elapsed = time.time() - t0
print(f"\nexit code {rc}; wall {elapsed/60:.1f} min", flush=True)

# GPU memory snapshot after run
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total", "--format=csv"],
                         capture_output=True, text=True).stdout)
except Exception as e:
    print("nvidia-smi failed", e)

out_w = TRAIN_REPO / "weights" / "prof" / "split_0"
if out_w.exists():
    dst = WORKING_DIR / "prof_weights"
    shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree(out_w, dst)
    print("saved weights:", [p.name for p in dst.iterdir()])

summary = {"extra_flags": os.environ.get("BIOHUB_PROF_EXTRA_FLAGS", ""), "epochs": EPOCHS, "max_iters": MAX_ITERS, "batch": BATCH, "n_train": len(train), "n_val": len(val),
           "window": WINDOW, "downsample": DOWNSAMPLE, "wall_min": elapsed / 60, "exit_code": rc}
(WORKING_DIR / "profile_summary.json").write_text(json.dumps(summary, indent=1))
print(summary)
if rc != 0:
    raise SystemExit(rc)
